# 🎬 ROBLOX AI SHORTS STUDIO — Google Colab + Google Drive

Crie vídeos verticais **9:16 (1080×1920)** de histórias de **Roblox em 3D (Blender + GPU CUDA)** com narração natural em PT-BR (**Kokoro TTS**), legendas sincronizadas (**Whisper GPU**), música instrumental e edição automática (**FFmpeg/NVENC**) — **salvando tudo automaticamente no seu Google Drive!**

### ⚡ Antes de começar (muito importante!)
1. No menu superior do Colab, clique em **Ambiente de execução → Alterar o tipo de ambiente de execução**.
2. Escolha **GPU T4** (gratuito) e clique em **Salvar**.
3. Execute a **Célula 1** (conectar Google Drive e clonar seu repositório do GitHub).
4. Execute a **Célula 2** (baixa e instala automaticamente o **Blender 5.3.0 Linux x64**, FFmpeg, motor 3D `rbx` já embutido no repositório e modelos leves — salvando cache no seu Drive para iniciar bem mais rápido nas próximas vezes!).
5. Escolha a **Célula 3 (Interface Web completa)** OU a **Célula 4 (Gerar vídeo em 1 clique direto no notebook)**.

📂 Todos os vídeos prontos aparecerão automaticamente no seu Google Drive em: `Meu Drive / RobloxAIShortsStudio / Videos_Prontos`.

In [ ]:
#@title 1️⃣ Conectar Google Drive e Clonar Repositório do GitHub
GITHUB_REPO_URL = "https://github.com/SEU_USUARIO/RobloxAIShortsStudio.git" #@param {type:"string"}
BRANCH = "main" #@param {type:"string"}
DRIVE_FOLDER_NAME = "RobloxAIShortsStudio" #@param {type:"string"}

import os
import subprocess
from pathlib import Path
from google.colab import drive

# 1. Monta o Google Drive na sua conta Google
drive.mount("/content/drive", force_remount=False)

DRIVE_ROOT = Path(f"/content/drive/MyDrive/{DRIVE_FOLDER_NAME}")
for sub in ("projects", "models", "Videos_Prontos"):
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)

# 2. Clona (ou atualiza) o repositório do GitHub
REPO_DIR = Path("/content/RobloxAIShortsStudio")
if "SEU_USUARIO" in GITHUB_REPO_URL:
    raise ValueError(
        "Troque 'SEU_USUARIO' em GITHUB_REPO_URL pelo link real do seu repositório no GitHub antes de executar!"
    )

if REPO_DIR.exists() and (REPO_DIR / ".git").exists():
    print("Atualizando repositório existente...")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", BRANCH], check=True)
else:
    print(f"Clonando {GITHUB_REPO_URL} ...")
    subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", GITHUB_REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)

# 3. Configura variáveis de ambiente para persistir projetos, modelos e vídeos no Google Drive
os.environ["STUDIO_COLAB_MODE"] = "1"
os.environ["STUDIO_PROJECTS_DIR"] = str(DRIVE_ROOT / "projects")
os.environ["STUDIO_MODELS_DIR"] = str(DRIVE_ROOT / "models")
os.environ["STUDIO_DRIVE_EXPORT_DIR"] = str(DRIVE_ROOT / "Videos_Prontos")
os.environ["DRIVE_ROOT"] = str(DRIVE_ROOT)

print("\n✅ Google Drive conectado e repositório pronto!")
print(f"📁 Projetos salvos em:      {os.environ['STUDIO_PROJECTS_DIR']}")
print(f"🧠 Cache de modelos em:     {os.environ['STUDIO_MODELS_DIR']}")
print(f"🎬 Vídeos prontos no Drive: {os.environ['STUDIO_DRIVE_EXPORT_DIR']}")

In [ ]:
#@title 2️⃣ Instalar Blender 3D, FFmpeg, Dependências e Modelos (com GPU T4 + Cache no Drive)
!nvidia-smi || echo "Aviso: sem GPU ativa (mude em Ambiente de execução -> Alterar tipo -> GPU T4 para renderizar muito mais rápido!)"
!bash /content/RobloxAIShortsStudio/scripts/colab_setup.sh

---
## Escolha como gerar seus vídeos:
- **OPÇÃO A (Célula 3):** Abre a **interface visual completa (Gradio)** com link público clicável (`https://xxxx.gradio.live`). Todos os vídeos gerados lá dentro vão direto pro seu Google Drive!
- **OPÇÃO B (Célula 4):** Gera o vídeo em **1 clique direto aqui no Colab**, sem abrir interface externa, e já mostra o player do vídeo pronto abaixo + salva no Drive!

In [ ]:
#@title 3️⃣ OPÇÃO A — Abrir Interface Web Completa (Link Público Gradio)
#@markdown Clique no link `Running on public URL: https://xxxx.gradio.live` que aparecer abaixo.
#@markdown Todos os vídeos gerados serão salvos automaticamente no seu Google Drive!
import os
os.chdir("/content/RobloxAIShortsStudio")
drive_root = os.environ.get("DRIVE_ROOT", "/content/drive/MyDrive/RobloxAIShortsStudio")
!python3 main.py --colab --drive-dir "{drive_root}"

In [ ]:
#@title 4️⃣ OPÇÃO B — Gerar Vídeo em 1 Clique Direto no Colab (e Salvar no Google Drive)
HISTORIA = "Neste servidor secreto de Roblox, todo jogador nasce com um número brilhando em cima da cabeça. O Lucas entrou achando que era um minigame comum, mas percebeu que quem falava a palavra 'sair' perdia metade da contagem na hora. Quando o relógio chegou perto de zero, um Admin Sombrio apareceu bloqueando a única ponte do mapa. No último segundo, Lucas descobriu que o número não era uma contagem de eliminação, mas a energia para abrir o portal de saída para todos os jogadores presos." #@param {type:"string"}
GENERO = "Suspense / Mistério" #@param ["Suspense / Mistério", "Terror Leve / Sobrevivência", "Aventura / Ação", "Reviravolta / Drama", "Comédia / Caos no Servidor", "Ficção Científica / Regra Secreta"]
ABERTURA = "Neste servidor..." #@param ["Neste servidor...", "Nesta cidade...", "Neste mundo...", "Nesta escola...", "Neste lugar, existia uma regra...", "Todos os jogadores precisavam..."]
DURACAO_SEGUNDOS = 55 #@param {type:"slider", min:45, max:60, step:1}
VOZ = "pm_alex" #@param ["pm_alex", "pf_dora", "pm_santa"]
VELOCIDADE = 1.05 #@param {type:"slider", min:0.85, max:1.30, step:0.05}
MUSICA = "suspense" #@param ["suspense", "divertida", "emocionante", "none"]
ESTILO_VISUAL = "roblox_3d_blender_cinematic" #@param ["roblox_3d_blender_cinematic", "roblox_3d_dark_mystery", "roblox_3d_vibrant_adventure", "roblox_3d_neon_cyber"]
GEMINI_API_KEY = "" #@param {type:"string"}

import base64
import os
import subprocess
from pathlib import Path
from IPython.display import HTML, display

os.chdir("/content/RobloxAIShortsStudio")
drive_root = os.environ.get("DRIVE_ROOT", "/content/drive/MyDrive/RobloxAIShortsStudio")

cmd = [
    "python3", "main.py",
    "--colab",
    "--drive-dir", drive_root,
    "--generate", HISTORIA,
    "--genre", GENERO,
    "--opening", ABERTURA,
    "--duration", str(DURACAO_SEGUNDOS),
    "--voice", VOZ,
    "--speed", str(VELOCIDADE),
    "--music", MUSICA,
    "--style", ESTILO_VISUAL,
]
if GEMINI_API_KEY.strip():
    cmd.extend(["--gemini-key", GEMINI_API_KEY.strip()])

subprocess.run(cmd, check=True)

# Exibe o vídeo mais recente salvo na pasta Videos_Prontos do Google Drive
export_dir = Path(drive_root) / "Videos_Prontos"
mp4s = sorted(export_dir.glob("*.mp4"), key=lambda p: p.stat().st_mtime, reverse=True)
if mp4s:
    latest = mp4s[0]
    print(f"\n🎉 Vídeo pronto salvo no seu Google Drive: {latest}")
    b64 = base64.b64encode(latest.read_bytes()).decode("ascii")
    display(HTML(
        f'<video width="360" controls style="border-radius:12px;border:2px solid #3b82f6;">'
        f'<source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'
    ))

In [ ]:
#@title 5️⃣ Ver Todos os Vídeos Prontos no Seu Google Drive
import os
from datetime import datetime
from pathlib import Path

drive_root = Path(os.environ.get("DRIVE_ROOT", "/content/drive/MyDrive/RobloxAIShortsStudio"))
export_dir = drive_root / "Videos_Prontos"
mp4s = sorted(export_dir.glob("*.mp4"), key=lambda p: p.stat().st_mtime, reverse=True)

if not mp4s:
    print(f"Nenhum vídeo encontrado ainda em {export_dir}")
else:
    print(f"📂 Pasta no Google Drive: {export_dir}\n")
    for p in mp4s:
        mb = p.stat().st_size / (1024 * 1024)
        dt = datetime.fromtimestamp(p.stat().st_mtime).strftime("%d/%m/%Y %H:%M")
        print(f"  🎬 {p.name}  ({mb:.1f} MB — {dt})")